# Step 1: Save CLIP Activations from ImageNet
Loads CLIP ViT-L/14 (from LLaVA-NeXT), passes ImageNet images through layer 22, saves activations to disk.
Memory-safe: processes in batches, saves each batch immediately, never holds everything in RAM.

In [1]:
!git clone https://github.com/ExplainableML/sae-for-vlm.git /kaggle/working/sae-for-vlm 2>/dev/null
!pip install -q -r /kaggle/working/sae-for-vlm/requirements.txt 2>/dev/null
!pip install -q huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.8/170.8 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 5.1 MB/s eta 0:00:00


In [2]:
import os
import sys
import glob
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

# ── CONFIG ──────────────────────────────────────────────────────────────────
IMAGENET_ROOT   = "/kaggle/input/datasets/vitaliykinakh/stable-imagenet1k/imagenet1k"
OUT_TRAIN_DIR   = "/kaggle/working/activations/train"
OUT_VAL_DIR     = "/kaggle/working/activations/val"

TARGET_LAYER    = 16          # CLIP ViT-L/14 has 24 layers (0-indexed), paper uses layer 22
BATCH_SIZE      = 64          # Lower if you get OOM. Try 32 if needed.
MAX_IMAGES      = 50_000      # Use None for all ~1.2M. 50k is good for training an SAE.
VAL_FRACTION    = 0.1         # 10% of collected images go to validation
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_EVERY_N_BATCHES = 10     # Save a chunk to disk every N batches (memory safety)
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_TRAIN_DIR, exist_ok=True)
os.makedirs(OUT_VAL_DIR, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Saving train activations → {OUT_TRAIN_DIR}")
print(f"Saving val activations → {OUT_VAL_DIR}")

Device: cuda
Saving train activations → /kaggle/working/activations/train
Saving val activations → /kaggle/working/activations/val


In [3]:
import os
from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient

print("Authenticating to Hugging Face...")
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# ── LOAD CLIP ViT-L/14 (same vision encoder used in LLaVA-NeXT) ─────────────
MODEL_NAME = "openai/clip-vit-large-patch14"
print(f"Loading {MODEL_NAME} ...")

processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
model = model.to(DEVICE)
model.eval()
print("Model loaded.")
print(f"Vision transformer has {len(model.vision_model.encoder.layers)} layers")

Authenticating to Hugging Face...
Loading openai/clip-vit-large-patch14 ...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.
Vision transformer has 24 layers


In [4]:
# ── HOOK to capture the output of layer TARGET_LAYER ────────────────────────
# The residual stream AFTER layer 22 = the output hidden state of that transformer block.
# Shape per image: [num_tokens, 1024]  (ViT-L/14 has 257 tokens: 1 CLS + 256 patch tokens)
# The paper uses CLS token only (index 0) or 2 random tokens.
# We'll use CLS token → gives us one 1024-dim vector per image.

_hook_output = []

def _hook_fn(module, input, output):
    # output is a tuple; first element is the hidden state tensor [batch, seq_len, dim]
    hidden = output[0] if isinstance(output, tuple) else output
    _hook_output.append(hidden[:, 0, :].detach().cpu().float())  # CLS token only

# Register the hook on the target layer
hook_handle = model.vision_model.encoder.layers[TARGET_LAYER].register_forward_hook(_hook_fn)
print(f"Hook registered on vision encoder layer {TARGET_LAYER}")
print(f"Activation dimension will be: {model.vision_model.config.hidden_size}")

Hook registered on vision encoder layer 16
Activation dimension will be: 1024


In [5]:
# ── COLLECT IMAGE PATHS ──────────────────────────────────────────────────────
print("Scanning image paths...")
all_paths = sorted(glob.glob(os.path.join(IMAGENET_ROOT, "*", "*.jpg")))
print(f"Found {len(all_paths):,} images")

if MAX_IMAGES is not None and len(all_paths) > MAX_IMAGES:
    # Shuffle deterministically and take first MAX_IMAGES
    rng = np.random.default_rng(42)
    indices = rng.permutation(len(all_paths))[:MAX_IMAGES]
    all_paths = [all_paths[i] for i in sorted(indices)]
    print(f"Subsampled to {len(all_paths):,} images")

# Train / val split
n_val = int(len(all_paths) * VAL_FRACTION)
val_paths   = all_paths[:n_val]
train_paths = all_paths[n_val:]
print(f"Train: {len(train_paths):,}  |  Val: {len(val_paths):,}")

Scanning image paths...
Found 100,001 images
Subsampled to 50,000 images
Train: 45,000  |  Val: 5,000


In [6]:
# ── EXTRACTION FUNCTION ──────────────────────────────────────────────────────
def extract_and_save(image_paths, out_dir, split_name):
    """Pass images through CLIP in batches, collect CLS activations, save chunks to disk."""
    chunk_buffer = []   # accumulate tensors before saving a chunk
    chunk_idx    = 0
    total_saved  = 0

    n_batches = (len(image_paths) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_num in tqdm(range(n_batches), desc=f"Extracting [{split_name}]"):
        batch_paths = image_paths[batch_num * BATCH_SIZE : (batch_num + 1) * BATCH_SIZE]

        # Load images — skip broken ones
        images = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert("RGB")
                images.append(img)
            except Exception:
                pass

        if not images:
            continue

        # Preprocess
        inputs = processor(images=images, return_tensors="pt", padding=True)
        pixel_values = inputs["pixel_values"].to(DEVICE, dtype=torch.float16)

        # Forward pass — hook captures the layer output
        _hook_output.clear()
        with torch.no_grad():
            model.vision_model(pixel_values=pixel_values)

        # _hook_output[0] shape: [batch, 1024]
        activations = _hook_output[0]   # already on CPU, float32
        chunk_buffer.append(activations)

        # Save chunk to disk every SAVE_EVERY_N_BATCHES batches
        if (batch_num + 1) % SAVE_EVERY_N_BATCHES == 0 or batch_num == n_batches - 1:
            chunk_tensor = torch.cat(chunk_buffer, dim=0)  # [N, 1024]
            save_path = os.path.join(out_dir, f"chunk_{chunk_idx:05d}.pt")
            torch.save(chunk_tensor, save_path)
            total_saved += chunk_tensor.shape[0]
            chunk_buffer.clear()
            chunk_idx += 1

            # Free memory
            del chunk_tensor
            torch.cuda.empty_cache()

    print(f"[{split_name}] Done. Saved {total_saved:,} activation vectors in {chunk_idx} chunks → {out_dir}")
    return total_saved

In [7]:
# ── RUN EXTRACTION ───────────────────────────────────────────────────────────
extract_and_save(train_paths, OUT_TRAIN_DIR, "train")
extract_and_save(val_paths,   OUT_VAL_DIR,   "val")

# Remove hook
hook_handle.remove()
print("\nHook removed.")

Extracting [train]: 100%|██████████| 704/704 [18:34<00:00,  1.58s/it]


[train] Done. Saved 45,000 activation vectors in 71 chunks → /kaggle/working/activations/train


Extracting [val]: 100%|██████████| 79/79 [02:12<00:00,  1.68s/it]

[val] Done. Saved 5,000 activation vectors in 8 chunks → /kaggle/working/activations/val

Hook removed.


In [8]:
# ── VERIFY: Check saved files ────────────────────────────────────────────────
train_chunks = sorted(glob.glob(os.path.join(OUT_TRAIN_DIR, "*.pt")))
val_chunks   = sorted(glob.glob(os.path.join(OUT_VAL_DIR,   "*.pt")))

print(f"Train chunks: {len(train_chunks)}")
print(f"Val   chunks: {len(val_chunks)}")

if train_chunks:
    sample = torch.load(train_chunks[0])
    print(f"\nSample chunk shape: {sample.shape}  dtype: {sample.dtype}")
    print(f"Activation dim: {sample.shape[1]}  (should be 1024 for CLIP ViT-L/14)")

# Free model memory — done with CLIP
del model, processor
torch.cuda.empty_cache()
import gc; gc.collect()
print("\nCLIP model freed from memory. Ready for SAE training.")

Train chunks: 71
Val   chunks: 8

Sample chunk shape: torch.Size([640, 1024])  dtype: torch.float32
Activation dim: 1024  (should be 1024 for CLIP ViT-L/14)

CLIP model freed from memory. Ready for SAE training.


In [9]:
# ── ZIP AND DOWNLOAD OUTPUT ──────────────────────────────────────────────────
import shutil
from IPython.display import FileLink

print("Zipping the activations folder. This might take a minute depending on the size...")

# This creates a file named 'activations_archive.zip' containing everything in the activations folder
shutil.make_archive('/kaggle/working/activations_archive', 'zip', '/kaggle/working/activations')

print("Zipping complete! Click the link below to download:")

# Generates a clickable download link in your Kaggle notebook
FileLink(r'activations_archive.zip')

Zipping the activations folder. This might take a minute depending on the size...
Zipping complete! Click the link below to download:


/kaggle/working/activations_archive.zip